# Problem Book P2, Chapter 1: Almost Every Number (Python companion)

Part A checks the numbers of the chapter, problem by problem. Part B is a laboratory: three experiments that push the problems further, with exercises first and
solutions at the end. We use `mpmath` for high precision (a random real number is simulated by $3600$ random digits, enough for its first $3000$
continued fraction digits, since the $n$-th one needs about $2n\log_{10}e^{1.19}\approx1.03\,n$ digits), `numpy` for linear algebra and `sympy`
for primes. The whole notebook runs in a few minutes.

*Before running a cell, predict its output.*

In [1]:
import math, random
from math import log, sqrt, pi, exp
import numpy as np
import mpmath as mp
from sympy import primerange
random.seed(1)
mp.mp.dps = 4000

def cf(x, n):
    # first n partial quotients a_0, a_1, ... of x
    a = []
    for _ in range(n):
        ai = int(mp.floor(x)); a.append(ai); x = 1 / (x - ai)
    return a

def convergents(a):
    p0, q0, p1, q1 = 1, 0, a[0], 1
    out = [(p1, q1)]
    for ai in a[1:]:
        p0, q0, p1, q1 = p1, q1, ai * p1 + p0, ai * q1 + q0
        out.append((p1, q1))
    return out

def random_real(digits=3600):
    return mp.mpf(random.getrandbits(int(digits * 3.33))) / mp.mpf(2) ** int(digits * 3.33)

## Part A: checks

**Problem 1.2 (badly approximable numbers).** For $\sqrt2$, with digits bounded by $K=2$, the solution gives $q^2|\sqrt2-\frac pq|\ge\frac1{K+2}=\frac14$. Check it for all $q\le10^5$.

In [2]:
s2 = mp.sqrt(2)
m = min(q * abs(q * s2 - mp.nint(q * s2)) for q in range(1, 10 ** 5 + 1))
print("min q^2 |sqrt2 - p/q| for q <= 1e5:", mp.nstr(m, 8), ">= 1/4:", m >= 0.25)
print("digits of sqrt 3:", cf(mp.sqrt(3), 12), " of (1+sqrt 13)/2:", cf((1 + mp.sqrt(13)) / 2, 12))

min q^2 |sqrt2 - p/q| for q <= 1e5: 0.34314575 >= 1/4: True
digits of sqrt 3: [1, 1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1]  of (1+sqrt 13)/2: [2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]


**Problem 1.3(a) (bounded distortion).** The conditional probability that $a_{n+1}\ge k$ given a cylinder is $\frac{\lambda(\psi(0,1/k))}{\lambda(I)}$, computable exactly. Check that
it lies in $[\frac1{2k},\frac2k]$ for random cylinders.

In [3]:
from fractions import Fraction as F
worst = (10, 0)
for trial in range(2000):
    n = random.randint(1, 8); w = [random.randint(1, 30) for _ in range(n)]
    p0, q0, p1, q1 = 1, 0, 0, 1
    for a in w: p0, q0, p1, q1 = p1, q1, a * p1 + p0, a * q1 + q0
    psi = lambda t: (F(p1) + t * p0) / (F(q1) + t * q0)
    k = random.randint(1, 50)
    r = abs(psi(F(1, k)) - psi(F(0))) / abs(psi(F(1)) - psi(F(0)))
    worst = (min(worst[0], float(r * k)), max(worst[1], float(r * k)))
print("k * P(a_{n+1} >= k | I) ranges over", worst, "(must lie in [1/2, 2])")

k * P(a_{n+1} >= k | I) ranges over (1.0, 1.9591836734693877) (must lie in [1/2, 2])


**Problems 1.3 and 1.4 (large digits; Khinchin).** For ten random numbers: the largest $\frac{a_n}{n\log n}$ for $n\le3000$, the denominators against $e^{2n}$, and the number of
$n$ with $a_{n+1}\ge\log q_n$, the condition behind $|x-\frac{p_n}{q_n}|<\frac1{q_n^2\log q_n}$.

In [4]:
for t in range(10):
    x = random_real(); a = cf(x, 3001)[1:]
    q0, q1, cnt, worst = 0, 1, 0, 0
    for n, an in enumerate(a, start=1):
        q0, q1 = q1, an * q1 + q0              # q1 = q_n
        if n < len(a) and n > 1 and a[n] >= log(q1): cnt += 1
    big = max(a[n - 1] / (n * log(n)) for n in range(2, 3001))
    print("x%d: max a_n/(n log n) = %.2f,  log(q_3000)/3000 = %.4f,  #{n: a_(n+1) >= log q_n} = %d" % (t, big, math.log(q1) / 3000, cnt))
print("Levy's constant pi^2/(12 log 2) =", pi ** 2 / (12 * log(2)))

x0: max a_n/(n log n) = 0.99,  log(q_3000)/3000 = 1.1987,  #{n: a_(n+1) >= log q_n} = 10


x1: max a_n/(n log n) = 1.21,  log(q_3000)/3000 = 1.1651,  #{n: a_(n+1) >= log q_n} = 7


x2: max a_n/(n log n) = 1.80,  log(q_3000)/3000 = 1.1588,  #{n: a_(n+1) >= log q_n} = 10


x3: max a_n/(n log n) = 6.49,  log(q_3000)/3000 = 1.1869,  #{n: a_(n+1) >= log q_n} = 8


x4: max a_n/(n log n) = 7.81,  log(q_3000)/3000 = 1.2089,  #{n: a_(n+1) >= log q_n} = 10


x5: max a_n/(n log n) = 1.59,  log(q_3000)/3000 = 1.1637,  #{n: a_(n+1) >= log q_n} = 8


x6: max a_n/(n log n) = 5.48,  log(q_3000)/3000 = 1.1727,  #{n: a_(n+1) >= log q_n} = 7


x7: max a_n/(n log n) = 6.99,  log(q_3000)/3000 = 1.1562,  #{n: a_(n+1) >= log q_n} = 14


x8: max a_n/(n log n) = 2.89,  log(q_3000)/3000 = 1.1998,  #{n: a_(n+1) >= log q_n} = 7


x9: max a_n/(n log n) = 13.96,  log(q_3000)/3000 = 1.1768,  #{n: a_(n+1) >= log q_n} = 5
Levy's constant pi^2/(12 log 2) = 1.1865691104156255


**Problem 1.5 ($e$).** The digits, the eight solutions of $|e-\frac pq|<\frac1{q^2\log q}$ with $q\ge2$, and the proof that there are no more: beyond the $60$-th convergent,
$\log q_n>a_{n+1}+2$, so $q^2\log q\,|e-\frac pq|>\frac{\log q_n}{a_{n+1}+2}>1$ for every $q\ge q_n$ in the range of that convergent.

In [5]:
mp.mp.dps = 6000
a = cf(mp.e, 2002); print(a[:16])
assert all(a[m] == (2 * (m + 1) // 3 if m % 3 == 2 else 1) for m in range(1, 2002))
P = convergents(a)
sols = [(p, q) for q in range(2, 8) for p in [int(mp.nint(q * mp.e))] if q * q * mp.log(q) * abs(mp.e - mp.mpf(p) / q) < 1]
sols += [(p, q) for (p, q) in P[1:] if q >= 8 and q * q * mp.log(q) * abs(mp.e - mp.mpf(p) / q) < 1]
print(sorted(set(sols), key=lambda t: t[1]))
print("log q_n > a_(n+1) + 2 for 60 <= n <= 2000:", all(mp.log(P[n][1]) > a[n + 1] + 2 for n in range(60, 2001)))
print("log q_2000 / ((2000/3) log 2000) =", float(mp.log(P[2000][1]) / (2000 / 3 * log(2000))))
mp.mp.dps = 4000

[2, 1, 2, 1, 1, 4, 1, 1, 6, 1, 1, 8, 1, 1, 10, 1]


[(5, 2), (8, 3), (11, 4), (19, 7), (193, 71), (2721, 1001), (49171, 18089), (1084483, 398959)]


log q_n > a_(n+1) + 2 for 60 <= n <= 2000: True
log q_2000 / ((2000/3) log 2000) = 0.9081938730524259


**Problem 1.6 (Minkowski's $?$).** The values at $\frac{\sqrt5-1}2$ and $\sqrt2-1$, monotonicity on random pairs, and the ratio $?(I)/\lambda(I)$ for the cylinders around a random
number.

In [6]:
def qmark(x, n=200):
    a = cf(x, n + 1)[1:]; s, tot = 0, mp.mpf(0)
    for k, ak in enumerate(a, start=1):
        s += ak; tot += 2 * (-1) ** (k + 1) * mp.mpf(2) ** (-s)
    return tot
print(mp.nstr(qmark((mp.sqrt(5) - 1) / 2), 15), mp.nstr(qmark(mp.sqrt(2) - 1), 15))
ok = True
for _ in range(300):
    u, v = sorted([random_real(60), random_real(60)])
    ok &= qmark(u, 60) < qmark(v, 60)
print("? increasing on 300 random pairs:", ok)
x = random_real(); a = cf(x, 401)[1:]
q0, q1, s = 0, 1, 0
for n, an in enumerate(a, start=1):
    q0, q1 = q1, an * q1 + q0; s += an
    if n in (10, 50, 100, 200, 400):
        print("n=%3d  s_n/n=%.2f  log10(?(I)/|I|) = %.1f" % (n, s / n, (math.log(q1 * (q1 + q0)) - s * log(2)) / log(10)))

0.666666666666667 0.4


? increasing on 300 random pairs: True
n= 10  s_n/n=18.40  log10(?(I)/|I|) = -44.1
n= 50  s_n/n=9.32  log10(?(I)/|I|) = -83.5
n=100  s_n/n=7.69  log10(?(I)/|I|) = -122.9
n=200  s_n/n=8.01  log10(?(I)/|I|) = -270.9
n=400  s_n/n=7.33  log10(?(I)/|I|) = -480.6


**Problem 1.8(b) (fractions with prime denominators).** For $Q=10^4$ and an interval $J$ of length $L=10^{-3}$, count the fractions $\frac pq\in J$ with $q$ prime in $]Q,2Q]$, against
the bound $\frac{LQ^2}{4\log Q}$, and the minimal spacing against $\frac1{4Q^2}$.

In [7]:
Q, L, c = 10 ** 4, 1e-3, 0.3141
fr = sorted(F(p, q) for q in primerange(Q + 1, 2 * Q + 1) for p in range(math.ceil(c * q), math.floor((c + L) * q) + 1))
print(len(fr), "fractions; bound", L * Q * Q / (4 * log(Q)))
print("min spacing * 4Q^2 =", float(min(b - a_ for a_, b in zip(fr, fr[1:])) * 4 * Q * Q))

15427 fractions; bound 2714.3405118953237
min spacing * 4Q^2 = 1.0175529693367102


**Problem 1.10 ($E_{\{1,2\}}$).** The hull, the gap $g=\frac2{\sqrt3}-1$, the two bounds, and the dimension as the zero of $\log\rho(s)$, where $\rho(s)$ is the spectral radius of
$\mathcal L_s$, discretised by collocation at $40$ Chebyshev points.

In [8]:
al, be = (sqrt(3) - 1) / 2, sqrt(3) - 1
print("hull", al, be, " gap", 1 / (1 + be) - 1 / (2 + al), 2 / sqrt(3) - 1)
print("bounds", log(2) / (2 * log((1 + sqrt(5)) / 2)), log(2) / (2 * log(1 + sqrt(2))))
def rho(s, digits=(1, 2), n=40):
    k = np.arange(n); x = 0.5 * (1 + np.cos(np.pi * (k + 0.5) / n))
    w = np.array([1 / np.prod([x[i] - x[j] for j in range(n) if j != i]) for i in range(n)])
    def lag(t):        # Lagrange basis values at the points t
        V = np.ones((len(t), n))
        for i in range(n):
            for j in range(n):
                if j != i: V[:, i] *= (t - x[j])
        return V * w
    M = sum(((a + x) ** (-2 * s))[:, None] * lag(1 / (a + x)) for a in digits)
    return max(abs(np.linalg.eigvals(M)))
def dim(digits, lo=0.0, hi=1.0):
    for _ in range(55):
        m = (lo + hi) / 2
        lo, hi = (m, hi) if rho(m, digits) > 1 else (lo, m)
    return lo
print("dim E_{1,2} =", dim((1, 2)))

hull 0.3660254037844386 0.7320508075688772  gap 0.15470053837925157 0.15470053837925168
bounds 0.7202100452062782 0.39321985067869747
dim E_{1,2} = 0.5312805062772057


**Problem 1.11 (Besicovitch--Eggleston).** $\frac{H(1/3)}{\log2}=\log_23-\frac23$.

In [9]:
H = lambda p: -p * log(p) - (1 - p) * log(1 - p)
print(H(1 / 3) / log(2), log(3, 2) - 2 / 3, " exponents:", H(1 / 3), log(2))

0.9182958340544894 0.9182958340544897  exponents: 0.6365141682948128 0.6931471805599453


**Problem 1.12 (Liouville numbers in the Cantor set).** For $x=\sum_k2\cdot3^{-k!}$, check $|x-\frac pq|\le q^{-n}$ with $q=3^{n!}$.

In [10]:
x = sum(2 * mp.mpf(3) ** (-math.factorial(k)) for k in range(1, 8))
for n in range(1, 6):
    q = mp.mpf(3) ** math.factorial(n); p = mp.nint(q * sum(2 * mp.mpf(3) ** (-math.factorial(k)) for k in range(1, n + 1)))
    print(n, mp.nstr(abs(x - p / q) * q ** n, 5), "<= 1")

1 0.6749 <= 1
2 0.22222 <= 1
3 0.0027435 <= 1
4 7.0814e-12 <= 1
5 1.113e-57 <= 1


**Problems 1.13 and 1.14.** $\{n!\,x\}$ for a random $x$ (equidistributed) and $\{n!\,e\}$, which lies in $]\frac1{n+1},\frac1n]$.

In [11]:
x = random_real(); v = x; c = [0] * 10
for n in range(1, 801):
    v = v * n if n > 1 else v
    c[int((v - mp.floor(v)) * 10)] += 1
print("deciles of {n! x}, n <= 800:", c)
fe = [mp.factorial(n) * mp.e for n in range(2, 12)]
print([(n, mp.nstr(t - mp.floor(t), 6), mp.nstr(mp.mpf(1) / n, 6)) for n, t in zip(range(2, 12), fe)])

deciles of {n! x}, n <= 800: [84, 71, 72, 77, 74, 110, 74, 84, 64, 90]
[(2, '0.436564', '0.5'), (3, '0.309691', '0.333333'), (4, '0.238764', '0.25'), (5, '0.193819', '0.2'), (6, '0.162916', '0.166667'), (7, '0.140415', '0.142857'), (8, '0.123323', '0.125'), (9, '0.109911', '0.111111'), (10, '0.0991122', '0.1'), (11, '0.090234', '0.0909091')]


**Problem 1.16 (Koksma).** The histograms of Figure 1.2.

In [12]:
mp.mp.dps = 1800
for name, x in (("1.37", mp.mpf("1.37")), ("phi", (1 + mp.sqrt(5)) / 2)):
    v, c = mp.mpf(1), [0] * 20
    for n in range(1, 4001):
        v *= x; c[min(19, int((v - mp.floor(v)) * 20))] += 1
    print(name, c)
mp.mp.dps = 4000

1.37 [213, 206, 215, 192, 202, 185, 178, 207, 212, 174, 188, 213, 209, 204, 195, 197, 196, 222, 194, 198]


phi [1997, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 1, 1, 1997]


**Problem 1.17 (the Cantor measure).** $|\hat\mu(1)|$, $|\hat\mu(3^m)|$, and a Monte Carlo estimate of $\hat\mu(1)$ from random points of $C$.

In [13]:
mu = lambda t: mp.e ** (-1j * pi * t) * mp.fprod(mp.cos(2 * pi * t / mp.mpf(3) ** k) for k in range(1, 120))
print([mp.nstr(abs(mu(3 ** m)), 10) for m in range(5)], mp.nstr(abs(mu(2)), 10))
X = np.array([sum(2 * random.getrandbits(1) * 3.0 ** (-k) for k in range(1, 40)) for _ in range(20000)])
print("Monte Carlo |mu^(1)| ~", abs(np.exp(-2j * pi * X).mean()))

['0.3714373567', '0.3714373567', '0.3714373567', '0.3714373567', '0.3714373567'] 0.07654171273
Monte Carlo |mu^(1)| ~ 0.37261289100395517


## Part B: laboratory

**L1.1 (Khinchin's constant against $e$ and $\pi$).** Compute $(a_1\cdots a_n)^{1/n}$ for $n=2000$ for five random numbers, for $\pi$ and for $e$, and compare with
$K=2.68545\ldots$ What do you predict for $e$, and why can no such computation decide whether $\pi$ is Khinchin-typical?

**L1.2 (more digits).** Compute $\dim_HE_{\{1,2,3\}}$, $\dim_HE_{\{1,\dots,10\}}$ and $\dim_HE_{\{2,3\}}$ with the operator of Problem 1.10, and compare $\dim_HE_{\{1,\dots,N\}}$ with Hensley's
$1-\frac6{\pi^2N}$.

**L1.3 (Borel--Bernstein by simulation).** Let $\varphi(n)=n\log n$. For $200$ random numbers, count the $n\le2000$ with $a_n\ge\varphi(n)$. Compare the mean count with
$\sum_n\lambda(a_n\ge\varphi(n))$, using the Gauss measure: $\gamma(a_n\ge k)=\log_2\frac{k+1}k$, and $\lambda(a_n\ge k)\to\gamma(a_1\ge k)$ as $n\to\infty$ (Tome 10, Chapter 10).

### Solutions

In [14]:
gm = lambda a: exp(sum(map(log, a)) / len(a))
for t in range(5):
    print("random", round(gm(cf(random_real(), 2001)[1:]), 4))
print("pi", round(gm(cf(mp.pi, 2001)[1:]), 4), " e", round(gm(cf(mp.e, 2001)[1:]), 4))

random 2.572


random 2.7005


random 2.6271


random 2.7606


random 2.6782


pi 2.6611  e 7.9124


Random numbers give values near $2.68$, as does $\pi$ (Tome 10's theorem predicts $K$ almost surely, and $\pi$ behaves as if random). For $e$ the value is about $5.9$ and keeps
growing: two digits in three are $1$ and the others are $2,4,6,\dots$, so $a_1\cdots a_n\approx2^{n/3}(n/3)!$ and $(a_1\cdots a_n)^{1/n}$ tends to infinity like a constant times $n^{1/3}$.
No finite computation decides a limit: an exceptional null set contains numbers that look typical up to any given $n$, so the value for $\pi$ is evidence, not proof.

In [15]:
for D in ((1, 2, 3), tuple(range(1, 11)), (2, 3)):
    print(D, round(dim(D), 6))
for N in (5, 10, 20):
    print(N, round(dim(tuple(range(1, N + 1))), 5), round(1 - 6 / (pi ** 2 * N), 5))

(1, 2, 3) 0.705661


(1, 2, 3, 4, 5, 6, 7, 8, 9, 10) 0.925738
(2, 3) 0.337437


5 0.83683 0.87841


10 0.92574 0.93921


20 0.96539 0.9696


$\dim_HE_{\{1,2,3\}}\approx0.7057$ and $\dim_HE_{\{2,3\}}\approx0.3370$ (no digit $1$ makes the cylinders much smaller). Hensley's formula is close already for $N=10$ and
improves with $N$; the next term in his expansion is of order $\frac{\log N}{N^2}$. (With $40$ collocation points the values are accurate to many digits for few digits;
for large $N$ more points help.)

In [16]:
mp.mp.dps = 2600
counts = []
for t in range(200):
    a = cf(random_real(2500), 2001)[1:]
    counts.append(sum(1 for n in range(2, 2001) if a[n - 1] >= n * log(n)))
pred = sum(log((math.ceil(n * log(n)) + 1) / math.ceil(n * log(n)), 2) for n in range(2, 2001))
print("mean count", np.mean(counts), "+-", np.std(counts) / sqrt(200), " predicted", pred)
mp.mp.dps = 4000

mean count 3.295 +- 0.10483260466095459  predicted 3.396616149846706


The mean count agrees with the prediction (about $3.4$) within the statistical error, and since $\sum\frac1{n\log n}$ diverges like $\log\log n$, the expected number of large digits keeps growing, very slowly:
this is the divergence half of Borel--Bernstein at work. With $\varphi(n)=n(\log n)^2$ the counts would stay bounded.